# Anomaly Detection on AI-Agent Access Behavior

**Author:** Jay Ekwere  
**Background:** Security engineer (vulnerability management @ Kinaxis; technical support on BeyondTrust's Pathfinder identity-security stack, which now governs AI-agent identities). This project applies ML to a problem adjacent to my day work: spotting when an AI agent's access behavior looks anomalous.

## The problem

AI agents increasingly act with real privileges — reading resources, calling APIs, escalating access. Most of that behavior is routine. The security question: **which sequences of actions look anomalous**, signalling a compromised, misconfigured, or misbehaving agent? This is the kind of question identity-security platforms (e.g. BeyondTrust Pathfinder) are now built to answer.

## Approach

Model *normal* agent behavior over short action sequences, then flag sequences that deviate. Baseline model: Isolation Forest (unsupervised, interpretable). Unit of analysis: rolling windows of actions, not single actions — because the meaningful anomalies are *patterns* (escalation chains, unusual ordering), not individual events.

## ⚠️ HONEST STATUS — this is a multi-day build in progress

**Day 1 (this notebook):** synthetic data generator + feature engineering + Isolation Forest baseline + evaluation scaffold.

**The hard part, not yet solved (Days 2–3):** the synthetic data is currently *too easy* — the injected anomalies are fairly obvious, so the model's good scores here are NOT evidence it would work on real logs. The real work is making the synthetic 'attacks' subtle enough to be a genuine test, then adding a second model (autoencoder) and comparing. Sections marked **`# TODO (harder)`** are where that work lives. I'd rather ship this honestly mid-build than overstate it.

Everything runs with standard libraries, no external data.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
print('Ready.')

## 1. Synthetic agent-action logs

We define a small vocabulary of agent actions and generate sessions. *Normal* sessions follow common patterns; *anomalous* sessions inject suspicious behavior (e.g. privilege escalation followed by bulk data access).

**Honest note:** the realism of this generator is the whole ballgame. Right now it's a first pass.

In [ ]:
ACTIONS = [
    'read_doc', 'list_resources', 'call_api', 'write_doc',
    'request_access', 'escalate_privilege', 'bulk_export', 'delete_resource'
]
A2I = {a: i for i, a in enumerate(ACTIONS)}

# Normal agents mostly read / list / call APIs, occasionally write.
NORMAL_WEIGHTS = np.array([0.30, 0.25, 0.25, 0.12, 0.05, 0.01, 0.01, 0.01])

def make_normal_session(length):
    return list(rng.choice(ACTIONS, size=length, p=NORMAL_WEIGHTS))

def make_anomalous_session(length):
    # TODO (harder): this anomaly is currently too 'loud' — a clear escalate->bulk_export->delete
    # chain. Real malicious agents blend in. Days 2-3: make anomalies subtler (e.g. only
    # slightly elevated escalation rate, anomalies spread across a session) so the model
    # is actually challenged. If detection stays near-perfect after that, the test is too easy.
    s = make_normal_session(length)
    inject_at = rng.integers(0, max(1, length - 3))
    s[inject_at:inject_at+3] = ['escalate_privilege', 'bulk_export', 'delete_resource']
    return s

N_NORMAL, N_ANOM = 2000, 100   # imbalanced, like reality
sessions, labels = [], []
for _ in range(N_NORMAL):
    L = int(rng.integers(8, 20)); sessions.append(make_normal_session(L)); labels.append(0)
for _ in range(N_ANOM):
    L = int(rng.integers(8, 20)); sessions.append(make_anomalous_session(L)); labels.append(1)
labels = np.array(labels)
print(f'{len(sessions)} sessions ({labels.sum()} anomalous, {100*labels.mean():.1f}%)')
print('Example normal: ', sessions[0][:8])
print('Example anomalous:', sessions[-1][:8])

## 2. Feature engineering over sequences

Turn each variable-length session into a fixed feature vector: action frequencies + a couple of simple sequence signals. 

**TODO (harder):** richer sequence features (n-gram transition probabilities, entropy, rare-transition counts) are where sequence modeling earns its keep. The current features are a starting set.

In [ ]:
def featurize(session):
    counts = np.zeros(len(ACTIONS))
    for a in session:
        counts[A2I[a]] += 1
    freq = counts / len(session)
    # simple sequence signals
    escalations = sum(1 for a in session if a == 'escalate_privilege')
    # count 'sensitive transitions': escalate immediately followed by export/delete
    sens_trans = sum(
        1 for i in range(len(session)-1)
        if session[i] == 'escalate_privilege' and session[i+1] in ('bulk_export', 'delete_resource')
    )
    return np.concatenate([freq, [escalations, sens_trans, len(session)]])

X = np.vstack([featurize(s) for s in sessions])
feat_names = ACTIONS + ['n_escalations', 'sensitive_transitions', 'session_len']
print('Feature matrix:', X.shape)

## 3. Isolation Forest baseline

Unsupervised: we train *without* labels (anomaly detection learns 'normal' and flags outliers), then use the held-out labels only to *evaluate*. `contamination` is set to the true anomaly rate — in production you wouldn't know this, which is itself a real limitation to discuss.

In [ ]:
Xs = StandardScaler().fit_transform(X)
iso = IsolationForest(contamination=N_ANOM/(N_NORMAL+N_ANOM), random_state=42, n_estimators=300)
iso.fit(Xs)
scores = -iso.score_samples(Xs)        # higher = more anomalous
preds = (iso.predict(Xs) == -1).astype(int)

print(f'ROC-AUC: {roc_auc_score(labels, scores):.4f}')
print(classification_report(labels, preds, digits=3))
print('Confusion matrix:\n', confusion_matrix(labels, preds))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(scores[labels==0], bins=40, alpha=0.6, label='normal')
ax[0].hist(scores[labels==1], bins=40, alpha=0.6, label='anomalous')
ax[0].set_title('Anomaly score distribution'); ax[0].set_xlabel('score'); ax[0].legend()

# crude feature signal: mean by class
means = pd.DataFrame(X, columns=feat_names).assign(label=labels).groupby('label').mean().T
means.plot.barh(ax=ax[1]); ax[1].set_title('Mean feature value by class')
plt.tight_layout(); plt.show()

## 4. Honest read & the real roadmap

**What to expect from Day 1:** detection here will likely look *very* good — and that is the warning sign, not the win. The injected anomalies are currently obvious (a loud escalate→export→delete chain), so the model catches them easily. That tells us almost nothing about real-world performance.

**Days 2–3 (the actual work):**
- Make anomalies subtle (see `# TODO (harder)` in section 1) until detection becomes genuinely hard — that's when the project means something.
- Add richer sequence features (transition probabilities, entropy).
- Add an autoencoder and compare it to Isolation Forest.
- Drop the `contamination`-knows-the-answer shortcut; test threshold selection without peeking at labels.

**Why this connects to my work:** identity-security platforms like BeyondTrust Pathfinder now map and monitor AI-agent privileges and access paths. This is a toy model of that detection problem — built to learn the ML side of something I already understand operationally.